In [70]:
import numpy as np
import tensorflow as tf

from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import StandardScaler

# Use your local filepath here
file_pattern = r'G:/.shortcut-targets-by-id/1abX3CWvYUSJM3cGg6r_GeHAVeNoYH6Ul/CS6140_Project_Data/koppen_shard_part_*.tfrecord.gz'
all_files = tf.io.gfile.glob(file_pattern)

In [71]:
def _parse_function(example_proto):
    # Define the specific keys found in the shards
    band_keys = ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B11', 'B12']

    feature_description = {
        'classification': tf.io.FixedLenFeature([1], tf.float32),
    }

    for key in band_keys:
        feature_description[key] = tf.io.FixedLenFeature([129 * 129], tf.float32)

    parsed_features = tf.io.parse_single_example(example_proto, feature_description)

    # Reconstruct the 12-channel tensor
    bands = []
    for key in band_keys:
        band = tf.reshape(parsed_features[key], [129, 129, 1])
        bands.append(band)

    # Stack along the last axis to get (129, 129, 12)
    full_tensor = tf.concat(bands, axis=-1)

    # Get the label
    label = tf.cast(parsed_features['classification'][0], tf.int64)

    return full_tensor, label

def load_shards(filenames):
    dataset = tf.data.TFRecordDataset(filenames, compression_type='GZIP')
    return dataset.map(_parse_function)

In [72]:
# Test the loader
loaded_shards = load_shards(all_files).batch(32)
labels = set()
for batch_tensors, batch_labels in loaded_shards.take(1).as_numpy_iterator():
    print(batch_tensors.shape)
    print(batch_labels.shape)

(32, 129, 129, 12)
(32,)


In [73]:
train_files, test_files = train_test_split(all_files, test_size=0.2, random_state=42)
print(f"Training on {len(train_files)} shards.")
print(f"Testing on {len(test_files)} shards.")

Training on 1200 shards.
Testing on 300 shards.


In [80]:
scaler = StandardScaler()
all_classes = np.arange(1, 31)
batch_size = 128
train_dataset = load_shards(train_files).batch(batch_size)

# --- Pass 1: Calculating scaling statistics ---
print("Pass 1: Calculating scaling statistics...")
for X_batch, _ in train_dataset.as_numpy_iterator():
    X_flat = X_batch.reshape(X_batch.shape[0], -1)
    scaler.partial_fit(X_flat)

print("Scaling statistics computed")

Pass 1: Calculating scaling statistics...
Scaling statistics computed


In [81]:
# Use SGDClassifier with loss='log_loss' to implement Logistic Regression with partial_fit
clf = SGDClassifier(
    loss='log_loss',
    penalty='l2',
    alpha=1e-5,
    random_state=42
)

# --- Pass 2: Model training ---
print('Pass 2: Training model...')
for i, (X_batch, y_batch) in enumerate(train_dataset.as_numpy_iterator()):
    X_flat = X_batch.reshape(X_batch.shape[0], -1)

    # Apply scaling
    X_scaled = scaler.transform(X_flat)

    # Fit model on current batch
    clf.partial_fit(X_scaled, y_batch, classes=all_classes)

    if i % 20 == 0:
        print(f"Trained on batch {i}")

print("Training completed")

Pass 2: Training model...
Trained on batch 0
Trained on batch 20
Trained on batch 40
Trained on batch 60
Trained on batch 80
Trained on batch 100
Trained on batch 120
Trained on batch 140
Trained on batch 160
Trained on batch 180
Trained on batch 200
Trained on batch 220
Trained on batch 240
Trained on batch 260
Trained on batch 280
Trained on batch 300
Trained on batch 320
Trained on batch 340
Trained on batch 360
Training completed


In [82]:
# --- Evaluation ---
print("Evaluating on training set...")
y_train_true, y_train_pred = [], []

for X_batch, y_batch in train_dataset.as_numpy_iterator():
    X_flat = X_batch.reshape(X_batch.shape[0], -1)
    X_scaled = scaler.transform(X_flat)

    y_train_pred.extend(clf.predict(X_scaled))
    y_train_true.extend(y_batch)

print("Evaluating on test set...")
test_ds = load_shards(test_files).batch(batch_size)
y_test_true, y_test_pred = [], []

for X_batch, y_batch in test_ds.as_numpy_iterator():
    X_flat = X_batch.reshape(X_batch.shape[0], -1)
    X_scaled = scaler.transform(X_flat)

    y_test_pred.extend(clf.predict(X_scaled))
    y_test_true.extend(y_batch)

Evaluating on training set...
Evaluating on test set...


In [83]:
print("Training classification report:")
print(classification_report(y_train_true, y_train_pred))
print('\n\n')
print("Test classification report:")
print(classification_report(y_test_true, y_test_pred))

Training classification report:
              precision    recall  f1-score   support

           1       0.26      0.01      0.02      1275
           2       0.04      0.00      0.00      1557
           3       0.12      0.00      0.00      1542
           4       0.13      0.04      0.06      1571
           5       0.13      0.00      0.01      1639
           6       0.01      0.00      0.00      1558
           7       0.00      0.00      0.00      1601
           8       0.00      0.00      0.00      1633
           9       0.10      0.00      0.00      1597
          10       0.05      0.00      0.00      1604
          11       0.00      0.00      0.00      1615
          12       0.05      0.00      0.00      1601
          13       0.23      0.00      0.01      1559
          14       0.02      0.06      0.03      1602
          15       0.17      0.09      0.12      1453
          16       0.05      0.01      0.02      1599
          17       0.30      0.10      0.15      